## Why Use RAG-Based Methodology Instead of Just LLMs?
Using a Retrieval-Augmented Generation (RAG) methodology offers significant advantages over traditional large language models (LLMs) alone:

#### Higher Accuracy and Reliability:
It retrieves factual information directly from different medical documents (books, papers), ensuring that responses are factually accurate and relevant. This drastically reduces the risk of generating incorrect information (known as "hallucinations"), which is a common issue when using LLMs on their own.

#### Targeted Knowledge Base:
Instead of relying on broad datasets, it focuses on a specific medical knowledge base, ensuring that users receive expert, specialized responses. This contrasts with general LLMs, which may provide superficial or irrelevant answers to health-related queries.

#### Contextual Relevance:
Through chunking and semantic embedding, it retrieves the most relevant information while maintaining the necessary context, which leads to clearer and more coherent responses tailored to the user's needs.

#### Efficient Memory Management:
Dividing the text into chunks avoids running into token limits of LLMs, allowing the model to process large documents effectively without losing important details from the encyclopedia.

#### Improved User Experience:
With precise, well-formatted responses, it enhances the readability and accessibility of complex medical information, making it easier for users to understand and act on the information provided.


#### Easy Knowledge Base Updates:
As medical knowledge evolves, it can easily be updated with new data, ensuring the chatbot remains current and provides the latest medical insights.

---
## Steps: 
### Document Reading with PyPDF:
The project utilizes the pyPDF library to read and extract data from different documents(books, papers). This library efficiently handles large documents, facilitating the structured extraction of valuable medical information.

### Data Chunking:
After extracting the data, it is crucial to divide the information into manageable chunks. Chunking helps to:

- Improve retrieval efficiency by enabling quick access to relevant sections.

- Enhance context provided in responses, allowing the model to generate more meaningful answers.

- Mitigate issues related to the maximum token limits of language models by ensuring that only concise and relevant information is processed at any given time.

### Creating Semantic Embeddings:

- To facilitate meaningful queries and responses, we employ the HuggingFaceEmbeddings model (sentence-transformers/all-MiniLM-L6-v2). This model generates semantic embeddings, which capture the context and meaning of the text beyond surface-level words. These embeddings enable GaleMed to comprehend user queries more effectively and retrieve the most relevant information.

### Building a Vector Database with Pinecone:
The next step involves creating a vector database using Pinecone. We load the chunked and embedded data into Pinecone, establishing a knowledge base that can efficiently manage user queries by identifying similar vectors and retrieving pertinent information.

### Testing Queries:
Once the knowledge base is established, we conduct test queries to evaluate the performance of GaleMed. This testing phase ensures that the chatbot retrieves accurate and relevant information effectively, confirming the effectiveness of the RAG approach.

### Next Step: Utilizing LLM for Response Formatting.



In [1]:
print("OK!")

OK!


In [2]:
import os
import time
from pinecone import Pinecone, ServerlessSpec
from langchain import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore as LangchainPinecone  # Updated import
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.llms import CTransformers


/Users/parthawgoswami/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/parthawgoswami/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/parthawgoswami/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Initializing index name and the Pinecone

os.environ["PINECONE_API_KEY"] = "pcsk_4ArCC1_Rzav3ZitfjUHYBhYFxYqkKBfVtqDyWUvH3ozBWD46nFViEpjBdx9djfBJJh8FLc"

index_name="rag-knowledge"

# Initialize Pinecone with optional parameters
try:
    pc = Pinecone(
        api_key=os.environ.get("PINECONE_API_KEY"),
        proxy_url=None,            # Example optional parameter
        proxy_headers=None,        # Example optional parameter
        ssl_ca_certs=None,        # Example optional parameter
        ssl_verify=True,  # Example optional parameter, usually set to True
    )
    
    # Check if the index exists
    indexes = pc.list_indexes()  # List of index names
    index_names = indexes.names()  # Get only the names of the indexes

    if index_name not in index_names:
        print(f'{index_name} does not exist')
        # Change the following line to create the index of your choice
        pc.create_index(
             name=index_name,
             dimension=384,
             metric="cosine",
             spec=ServerlessSpec(
                 cloud="aws",
                 region="us-east-1"
             )
         )
    else:
        print(f'{index_name} exists.')

    # Connect to the existing index
    index = pc.Index(index_name)

except Exception as e:
    print(f"An error occurred while checking indexes: {e}")

# Embedding the text chunks and storing them in Pinecone
try:
    docsearch = LangchainPinecone.from_texts(
        texts=[t.page_content for t in text_chunks],  # Assuming `text_chunks` is a list of text splits
        embedding=embeddings,  # Embedding model instance
        index_name=index_name
    )
except Exception as e:
    print(f"An error occurred while creating embeddings: {e}")

rag-knowledge exists.
An error occurred while creating embeddings: name 'text_chunks' is not defined


In [4]:
# from pinecone import Pinecone

pc = Pinecone(api_key="pcsk_4ArCC1_Rzav3ZitfjUHYBhYFxYqkKBfVtqDyWUvH3ozBWD46nFViEpjBdx9djfBJJh8FLc")
pc.list_indexes()

[
    {
        "name": "rag-knowledge",
        "metric": "cosine",
        "host": "rag-knowledge-oaavv06.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": null,
        "embed": {
            "model": "llama-text-embed-v2",
            "field_map": {
                "text": "text"
            },
            "dimension": 384,
            "metric": "cosine",
            "write_parameters": {
                "dimension": 384.0,
                "input_type": "passage",
                "truncate": "END"
            },
            "read_parameters": {
                "dimension": 384.0,
                "input_type": "query",
                "truncate": "END"
        

### Vector stores are there in the Pinecone that we created

In [5]:
#Extract data from the PDF
def load_pdf(data):
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)
    
    documents = loader.load()

    return documents

In [24]:
extracted_data = load_pdf("/Users/parthawgoswami/Documents/ADV_NLP/Medical_books/")

Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 51 0 (offset 0)
Ignoring wrong pointing object 96 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 51 0 (offset 0)
Ignoring wrong pointing object 96 0 (offset 0)
Ignoring wrong pointing object 274 0 (offset 0)
Ignoring wrong pointing object 988 0 (offset 0)
Ignoring wrong pointing object 274 0 (offset 0)
Ignoring wrong pointing object 988 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)
Ignoring wrong pointing object 49 0 (offset 0)
Ignoring w

In [7]:
#extracted_data

In [8]:
#Create text chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [25]:
text_chunks = text_split(extracted_data)
print("length of my chunk:", len(text_chunks))

length of my chunk: 18183


In [10]:
# text_chunks

In [26]:
#download embedding model
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [27]:
embeddings = download_hugging_face_embeddings()

In [28]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [29]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [15]:
# query_result

### Writing the Data into Pinecone

In [ ]:
#Initializing index name and the Pinecone

os.environ["PINECONE_API_KEY"] = "pcsk_4ArCC1_Rzav3ZitfjUHYBhYFxYqkKBfVtqDyWUvH3ozBWD46nFViEpjBdx9djfBJJh8FLc"

index_name="rag-knowledge"

# Initialize Pinecone with optional parameters
try:
    pc = Pinecone(
        api_key=os.environ.get("PINECONE_API_KEY"),
        proxy_url=None,            # Example optional parameter
        proxy_headers=None,        # Example optional parameter
        ssl_ca_certs=None,        # Example optional parameter
        ssl_verify=True,  # Example optional parameter, usually set to True
    )
    
    time.sleep(2)  # Optional sleep to ensure initialization completes

    # Check if the index exists
    indexes = pc.list_indexes()  # List of index names
    index_names = indexes.names()  # Get only the names of the indexes

    if index_name not in index_names:
        print(f'{index_name} does not exist')
        
    else:
        print(f'{index_name} exists.')

    # Connect to the existing index
    index = pc.Index(index_name)

except Exception as e:
    print(f"An error occurred while checking indexes: {e}")


rag-knowledge exists.


In [33]:
# Check how many vectors are in the index
stats = index.describe_index_stats()
print("Index stats:", stats)
print("Total vector count:", stats.get("total_vector_count", 0))

# If index is empty, upload text chunks
if stats.get("total_vector_count", 0) == 0:
    print(f"\nIndex is EMPTY! Uploading {len(text_chunks)} text chunks...")
    docsearch = LangchainPinecone.from_texts(
        texts=[t.page_content for t in text_chunks],
        embedding=embeddings,
        index_name=index_name
    )
    print("Upload complete! Waiting a few seconds for indexing...")
    time.sleep(5)
    # Re-check stats
    stats = index.describe_index_stats()
    print("Updated vector count:", stats.get("total_vector_count", 0))
else:
    print("Index already has vectors. Loading existing index...")
    docsearch = LangchainPinecone.from_existing_index(index_name, embeddings)


Index stats: {'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}
Total vector count: 0

Index is EMPTY! Uploading 18183 text chunks...
Upload complete! Waiting a few seconds for indexing...
Updated vector count: 18183


### Testing it out

In [34]:
index_name="rag-knowledge"
#If we already have an index we can load it like this
docsearch=LangchainPinecone.from_existing_index(index_name, embeddings)

query = "What are Allergies"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(id='d5dd4428-243e-45ca-a0e4-5e3557af2282', metadata={}, page_content='Purpose\nAllergy is a reaction of the immune system. Nor-\nmally, the immune system responds to foreign microor-\nganisms and particles, like pollen or dust, by producing\nspecific proteins called antibodies that are capable of\nbinding to identifying molecules, or antigens, on the\nforeign organisms. This reaction between antibody and\nantigen sets off a series of reactions designed to protect\nthe body from infection. Sometimes, this same series of'), Document(id='6c0949cb-75f2-4f3d-9b2c-e85a8ec267c0', metadata={}, page_content='reaction. Allergic rhinitis is characterized by an itchy,\nrunny nose, often with a scratchy or irritated throat due\nto post-nasal drip. Inflammation of the thin membrane\ncovering the eye (allergic conjunctivitis) causes redness,\nirritation, and increased tearing in the eyes. Asthma caus-\nes wheezing, coughing, and shortness of breath. Symp-\ntoms of food allergies depe

In [35]:
query = "Cure for acne?"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(id='a0b303d3-abf1-4532-8465-de7e3a124d94', metadata={}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'), Document(id='7c3808bc-c471-4d36-a163-81b791dc6615', metadata={}, page_content='milk thistle (Silybum marianum), and with nutrients such\nas essential fatty acids, vitamin B complex, zinc, vitamin\nA, and chromium is also recommended. Chinese herbal\nremedies used for acne include cnidium seed ( Cnidium\nmonnieri) and honeysuckle flower ( Lonicera japonica ).\nWholistic physicians or nutritionists can recommend the\nproper amounts of these herbs.\nPrognosis\nAcne is not curable, although long-term control is\nachieved in up to 60% of patients treated with'), Document(id='33b45390-e613-4b4d-a61b-588e29184f24', metadata={}, page_content='depends upon whether the acne is mild, moderate, or\nsevere.\nDrugs\nTOPICAL DRUGS. Treatment for mild noninflamma-\ntory acne consists of reducing the formation of new\ncomedo

In [36]:
query = "I have pain in my head"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(id='4504ae84-adfd-492d-bf0e-77b1e351df72', metadata={}, page_content='of the brain. They can rupture, causing subarachnoid hemorrhage.CLINICAL APPROACHHe a da c he  i s  one  of the  mos t c ommon c ompl a i nts  of pa ti e nts  i n me di c a l  pra c ti c e .  It periodically afflicts 90% of adults, and almost 25% have recurrent severe head-aches. As with many common symptoms, a broad range of conditions, from trivial to life-threatening, might be responsible. The majority of patients presenting with headache have tension-type, migraine, or cluster; however,'), Document(id='44908b33-ac41-42ac-aa04-8b4d591ff90d', metadata={}, page_content='A 5 9 -ye a r-o ld  w o m a n  co m e s t o  yo u r clin ic b e ca u se  sh e  is co n ce rn e d  t h a t  sh e  might have a brain tumor. She has had a fairly severe headache for the last  3 weeks (she rates it as an 8 on a scale of 1-10). She describes the pain as constant, occasionally throbbing but mostly a dull ache, and localiz

In [37]:
docs

[Document(id='4504ae84-adfd-492d-bf0e-77b1e351df72', metadata={}, page_content='of the brain. They can rupture, causing subarachnoid hemorrhage.CLINICAL APPROACHHe a da c he  i s  one  of the  mos t c ommon c ompl a i nts  of pa ti e nts  i n me di c a l  pra c ti c e .  It periodically afflicts 90% of adults, and almost 25% have recurrent severe head-aches. As with many common symptoms, a broad range of conditions, from trivial to life-threatening, might be responsible. The majority of patients presenting with headache have tension-type, migraine, or cluster; however,'),
 Document(id='44908b33-ac41-42ac-aa04-8b4d591ff90d', metadata={}, page_content='A 5 9 -ye a r-o ld  w o m a n  co m e s t o  yo u r clin ic b e ca u se  sh e  is co n ce rn e d  t h a t  sh e  might have a brain tumor. She has had a fairly severe headache for the last  3 weeks (she rates it as an 8 on a scale of 1-10). She describes the pain as constant, occasionally throbbing but mostly a dull ache, and localized to 

In [21]:
query = "I am sad all the time"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result []


In [38]:
query = "What to do if you are sad?"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(id='bf7c3e24-a459-41b1-abbe-a5f4e972db4f', metadata={}, page_content='diagnosed mild depression and offered therapy, but the girl failed to attend further sessions \nand was subsequently discharged back to the GP.\nExamination\nOn examination, the girl is well-groomed, quiet with her gaze lowered. Her answers to ques -\ntions on her well-being are monosyllabic. She denies suicidal intent or plans to harm herself. \nEnquiries on her daily activities are unfruitful as she does not engage in any conversation.'), Document(id='d6df2858-5103-458c-a32d-00ef515c06c1', metadata={}, page_content='knees and wrists are painful at all times and he has increasing difficulty in doing his work. \nHe is feeling very exhausted and tired, his mood is low and he becomes irritable very eas-\nily. He has been crying unprovoked. He admits to problems falling asleep at night and early \nmorning waking. He has daily thoughts of life not being worth living, but denies any suicidal \nthoughts or

In [39]:
query = "Who is superman?"

docs=docsearch.similarity_search(query, k=3)

print("Result", 
      docs)

Result [Document(id='a918edcb-48ed-4a8f-a905-837ec94bec0b', metadata={}, page_content='yourself.\nKey Points'), Document(id='f4969ac4-4814-4c95-abbd-c19dc5568104', metadata={}, page_content='subsequent career.\nP John Rees\nJames Pattison\nGwyn Williams'), Document(id='9debaf8f-deef-4685-aa98-5e1f357084b5', metadata={}, page_content='xv')]
